##Scorecard to indentify pertinent fields that need cleaned

In [0]:
# from pyspark.sql import functions as F

# table_name = "virtue_foundation_dataset.virtue_foundation_dataset.facilities"

# df = spark.table(table_name)

# total_rows = df.count()

# results = []

# for field in df.schema.fields:

#     col_name = field.name
#     dtype = field.dataType.simpleString()

#     metrics = {
#         "column_name": col_name,
#         "data_type": dtype
#     }

#     if dtype in ["string"]:

#         row = (
#             df.select(
#                 F.sum(F.when(F.col(col_name).isNull(), 1).otherwise(0)).alias("null_count"),

#                 F.sum(
#                     F.when(
#                         F.trim(F.col(col_name)) == "",
#                         1
#                     ).otherwise(0)
#                 ).alias("blank_count"),

#                 F.sum(
#                     F.when(
#                         F.lower(F.trim(F.col(col_name))) == "null",
#                         1
#                     ).otherwise(0)
#                 ).alias("literal_null_count"),

#                 F.sum(
#                     F.when(
#                         F.col(col_name).rlike(r"^\s*[\[\{].*[\]\}]\s*$"),
#                         1
#                     ).otherwise(0)
#                 ).alias("json_artifact_count"),

#                 F.approx_count_distinct(F.col(col_name)).alias("distinct_count"),

#                 F.min(F.length(F.col(col_name))).alias("min_length"),

#                 F.max(F.length(F.col(col_name))).alias("max_length")
#             )
#             .first()
#         )

#         metrics.update(row.asDict())

#     else:

#         row = (
#             df.select(
#                 F.sum(F.when(F.col(col_name).isNull(), 1).otherwise(0)).alias("null_count"),
#                 F.approx_count_distinct(F.col(col_name)).alias("distinct_count"),
#                 F.min(F.col(col_name)).alias("min_value"),
#                 F.max(F.col(col_name)).alias("max_value")
#             )
#             .first()
#         )

#         metrics.update(row.asDict())

#     metrics["total_rows"] = total_rows
#     metrics["null_pct"] = round(
#         100.0 * metrics.get("null_count", 0) / total_rows, 2
#     )

#     results.append(metrics)

# scorecard_df = spark.createDataFrame(results)

# display(
#     scorecard_df.orderBy(
#         F.desc("null_pct")
#     )
# )

## Scorecard to identify strings that need to be cleaned


In [0]:
# from pyspark.sql import functions as F

# display(
#     scorecard_df
#         .select(
#             "column_name",
#             "data_type",
#             "null_pct",
#             "null_count",
#             "blank_count",
#             "literal_null_count",
#             "json_artifact_count",
#             "distinct_count",
#             "min_length",
#             "max_length",
#             "min_value",
#             "max_value"
#         )
#         .orderBy(
#             F.desc("null_pct"),
#             F.desc("json_artifact_count")
#         )
# )

## Final Cleaning output

In [0]:
# from pyspark.sql import functions as F

# display(
#     spark.table(
#         "virtue_foundation_dataset.virtue_foundation_dataset.facilities"
#     )
#     .select(
#         "unique_id",
#         "name",
#         "organization_type",
#         "address_country",
#         "officialWebsite",
#         "latitude",
#         "longitude"
#     )
#     .filter(
#         F.col("name").isNull()
#         | F.col("latitude").isNull()
#         | F.col("longitude").isNull()
#     )
# )

In [0]:
# display(
#     df.select(
#         "name",
#         "organization_type",
#         "address_country"
#     )
#     .groupBy(
#         "name",
#         "organization_type",
#         "address_country"
#     )
#     .count()
#     .orderBy(F.desc("count"))
# )

## Clean Dataframe Based on Information Revealed Above

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ------------------------------------------------------------------------------
# Load source data
# ------------------------------------------------------------------------------

source_table = "virtue_foundation_dataset.bronze.facilities"
target_table = "virtue_foundation_dataset.silver.facilities_cleaned"

df = spark.table(source_table)

# ------------------------------------------------------------------------------
# Audit collector
# ------------------------------------------------------------------------------

audit_dfs = []

def capture_dropped(before_df, after_df, reason):
    return before_df.subtract(after_df).withColumn(
        "rejection_reason",
        F.lit(reason)
    )

def log_step(step_name, before_df, after_df, reason):
    before = before_df.count()
    after = after_df.count()

    print(step_name)
    print(f"  Before : {before:,}")
    print(f"  After  : {after:,}")
    print(f"  Dropped: {before - after:,}\n")

    audit_dfs.append(capture_dropped(before_df, after_df, reason))
    return after_df

# ------------------------------------------------------------------------------
# Step 0
# ------------------------------------------------------------------------------

df_step0 = df

# ------------------------------------------------------------------------------
# Step 1: UUID validation
# ------------------------------------------------------------------------------

valid_uuid = F.col("unique_id").rlike(
    "^[a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12}$"
)

df_step1 = df_step0.filter(valid_uuid)

df_step1 = log_step(
    "Step 1 - Invalid UUID removal",
    df_step0,
    df_step1,
    "invalid_uuid_format"
)

# ------------------------------------------------------------------------------
# Step 2: Name validation
# ------------------------------------------------------------------------------

valid_name = (
    F.col("name").isNotNull()
    & (F.trim(F.col("name")) != "")
    & ~F.lower(F.trim(F.col("name"))).isin("null", "[]", "{}")
)

df_step2 = df_step1.filter(valid_name)

df_step2 = log_step(
    "Step 2 - Invalid Name removal",
    df_step1,
    df_step2,
    "invalid_name"
)

# ------------------------------------------------------------------------------
# Step 3: Organization type validation
# ------------------------------------------------------------------------------

valid_org = F.lower(F.trim(F.col("organization_type"))) == "facility"

df_step3 = df_step2.filter(valid_org)

df_step3 = log_step(
    "Step 3 - Invalid Org Type removal",
    df_step2,
    df_step3,
    "invalid_org_type"
)

# ------------------------------------------------------------------------------
# Step 4: Coordinates validation
# ------------------------------------------------------------------------------

valid_coords = (
    # allow nulls to pass through
    (
        F.col("latitude").isNull()
        | F.col("longitude").isNull()
    )
    |
    (
        # otherwise must be valid + in India
        F.col("latitude").between(-90, 90)
        & F.col("longitude").between(-180, 180)
        & F.col("latitude").between(6.0, 37.6)
        & F.col("longitude").between(68.0, 97.5)
    )
)

df_step4 = df_step3.filter(valid_coords)

df_step4 = log_step(
    "Step 4 - Coordinates validation (NULLs allowed, India-only spatial filter)",
    df_step3,
    df_step4,
    "invalid_coordinates_or_outside_india"
)

df_step4 = df_step3.filter(valid_coords)

df_step4 = log_step(
    "Step 4 - Invalid Coordinates removal",
    df_step3,
    df_step4,
    "invalid_coordinates"
)

# ------------------------------------------------------------------------------
# Step 4b: Normalize ZIP (IMPORTANT FIX)
# ------------------------------------------------------------------------------

df_step4 = df_step4.withColumn(
    "address_zipOrPostCode",
    F.regexp_replace(F.col("address_zipOrPostCode"), r"\s+", "")
)

# ------------------------------------------------------------------------------
# Step 5: India + valid PIN code filter
# ------------------------------------------------------------------------------

valid_india_pin = (
    F.lower(F.trim(F.col("address_country"))) == "india"
) & (
    F.col("address_zipOrPostCode").rlike("^[1-9][0-9]{5}$")
)

df_step5 = df_step4.filter(valid_india_pin)

df_step5 = log_step(
    "Step 5 - India + valid PIN code filter",
    df_step4,
    df_step5,
    "not_india_or_invalid_pin"
)

# ------------------------------------------------------------------------------
# Step 6: Standardize strings
# ------------------------------------------------------------------------------

string_cols = [
    f.name for f in df_step5.schema.fields
    if isinstance(f.dataType, StringType)
]

df_step6 = df_step5
for c in string_cols:
    df_step6 = df_step6.withColumn(c, F.lower(F.trim(F.col(c))))

# ------------------------------------------------------------------------------
# Step 7: Deduplication (first pass)
# ------------------------------------------------------------------------------

df_step7 = df_step6.dropDuplicates([
    "name",
    "address_country",
    "latitude",
    "longitude"
])

df_step7 = log_step(
    "Step 6 - Deduplication (geo)",
    df_step6,
    df_step7,
    "duplicate_record"
)

# ------------------------------------------------------------------------------
# Step 8: Deduplication (address-level)
# ------------------------------------------------------------------------------

df_step8 = df_step7.dropDuplicates([
    "name",
    "address_country",
    "address_zipOrPostCode",
    "address_line1"
])

df_step8 = log_step(
    "Step 7 - Deduplication (address)",
    df_step7,
    df_step8,
    "duplicate_record"
)

# ------------------------------------------------------------------------------
# FINAL AUDIT TABLE
# ------------------------------------------------------------------------------

audit_df = audit_dfs[0]
for d in audit_dfs[1:]:
    audit_df = audit_df.unionByName(d, allowMissingColumns=True)

# ------------------------------------------------------------------------------
# Build final dataset with audit_reason
# ------------------------------------------------------------------------------

audit_lookup = audit_df.select("unique_id", "rejection_reason")

df_final = df_step8.join(
    audit_lookup,
    on="unique_id",
    how="left"
).withColumnRenamed(
    "rejection_reason",
    "audit_reason"
)

# Ensure clean rows explicitly null
df_final = df_final.withColumn(
    "audit_reason",
    F.when(F.col("audit_reason").isNull(), F.lit(None)).otherwise(F.col("audit_reason"))
)

# ------------------------------------------------------------------------------
# WRITE TO SILVER TABLE
# ------------------------------------------------------------------------------

df_final.write.mode("overwrite").saveAsTable(target_table)

# ------------------------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------------------------

print("\nFINAL SUMMARY")
print(f"Original Records : {df_step0.count():,}")
print(f"Final Records    : {df_step8.count():,}")
print(f"Dropped Records  : {audit_df.count():,}")

display(df_final)